In [1]:
from neo4j import GraphDatabase
import os
import json
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()  # will read .env in current dir

True

In [2]:
uri = os.getenv("NEO4J_URI")
user = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
DB  = os.getenv("NEO4J_DATABASE", "neo4j")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] 

driver = GraphDatabase.driver(uri, auth=(user, password))
print("[INFO] Connecting to:", uri)
print("[INFO] DB:", DB)
print("[INFO] USER:", user, "PWD:", password)
llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)

[INFO] Connecting to: neo4j+s://62b9e173.databases.neo4j.io
[INFO] DB: neo4j
[INFO] USER: neo4j PWD: oHiXwOTSqk3Tv9TkX-wrcngsjffZeIWIYkcvwwzBefQ


In [3]:
# Function to load JSON data
def load_knowledge_graph(json_file_path):
    with open(json_file_path, 'r') as file:
        return json.load(file)

In [4]:
def _sanitize_label(name: str) -> str:
    s = name.replace(" ", "_").replace("-", "_")
    s = "".join(ch for ch in s if ch.isalnum() or ch == "_")
    return f"`{s}`"

def create_node(tx, node):
    """
    Same id -> single node.
    - Add label(s) to the node (multi-label).
    - Merge other attributes.
    - Accumulate doc_id and title into arrays without duplicates.
    """
    attrs = dict(node.get("attributes", {}))

    doc_id = node.get("doc_id", attrs.pop("doc_id", None))
    title  = node.get("title",  attrs.pop("title",  None))

    raw_labels = node.get("label")
    if isinstance(raw_labels, str):
        labels_list = [raw_labels]
    elif isinstance(raw_labels, (list, tuple)):
        labels_list = list(raw_labels)
    else:
        labels_list = []

    # build :Label1:Label2 suffix
    labels_suffix = ""
    if labels_list:
        labels_suffix = ":" + ":".join(_sanitize_label(l) for l in labels_list if l)

    query = f"""
    // single node per id
    MERGE (n {{id: $id}})

    // add incoming labels (true multi-label)
    SET n{labels_suffix}

    // merge any other attributes (excluding doc_id/title)
    SET n += $attrs

    WITH n, $doc_id AS doc_id, $title AS title

    // init arrays if null
    SET n.doc_ids = CASE
        WHEN n.doc_ids IS NULL AND doc_id IS NOT NULL THEN [doc_id]
        WHEN n.doc_ids IS NULL THEN []
        ELSE n.doc_ids
    END,
    n.titles = CASE
        WHEN n.titles IS NULL AND title IS NOT NULL THEN [title]
        WHEN n.titles IS NULL THEN []
        ELSE n.titles
    END

    // append unique values (no duplicates)
    WITH n, doc_id, title
    SET n.doc_ids = CASE
        WHEN doc_id IS NULL OR doc_id IN n.doc_ids THEN n.doc_ids
        ELSE n.doc_ids + [doc_id]
    END,
    n.titles = CASE
        WHEN title IS NULL OR title IN n.titles THEN n.titles
        ELSE n.titles + [title]
    END
    """
    tx.run(query, id=node["id"], attrs=attrs, doc_id=doc_id, title=title)


In [5]:
def create_relationship(tx, relationship):
    if "type" not in relationship or "source" not in relationship or "target" not in relationship:
        print(f"Skipping relationship due to missing fields: {relationship}")
        return

    attributes = relationship.get("attributes", {})  # Likely empty in your new format
    attributes_str = ", ".join([f"{key}: ${key}" for key in attributes.keys()])
    rel_type = f"`{relationship['type'].replace(' ', '_').replace('-', '_')}`"

    query = f"""
    MATCH (a {{id: $source}}), (b {{id: $target}})
    MERGE (a)-[r:{rel_type}]->(b)
    {"SET r += {" + attributes_str + "}" if attributes_str else ""}
    """
    tx.run(query, source=relationship["source"], target=relationship["target"], **attributes)


In [6]:
def store_knowledge_graph(driver, graph):
    START_FROM_NODE_INDEX = 0
    with driver.session() as session:
        print("[INFO] Storing Nodes...")
        total_nodes = len(graph["nodes"])
        for idx, node in enumerate(graph["nodes"][START_FROM_NODE_INDEX:], start=START_FROM_NODE_INDEX + 1):
            if "id" not in node or "label" not in node:
                print(f"[WARNING] Skipping node with missing 'id' or 'label': {node}")
                continue
            print(f"[INFO] Adding Node {idx}/{total_nodes}: ID = {node.get('id')}, Label = {node.get('label')}")
            session.write_transaction(create_node, node)

        print("[INFO] Storing Relationships...")
        for idx, relationship in enumerate(graph["relationships"], start=1):
            print(f"[INFO] Adding Relationship {idx}/{len(graph['relationships'])}: Type = {relationship.get('type')}, Source = {relationship.get('source')}, Target = {relationship.get('target')}")
            session.write_transaction(create_relationship, relationship)



In [7]:
# Load the knowledge graph data from a JSON file
json_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/experiment/8_12k.json"   # Path to your JSON file
with open(json_file_path, "r") as file:
    knowledge_graph = json.load(file)

# Store the knowledge graph in Neo4j
try:
    store_knowledge_graph(driver, knowledge_graph)
    print("Knowledge graph stored in Neo4j successfully!")
finally:
    driver.close()

[INFO] Storing Nodes...
[INFO] Adding Node 1/31011: ID = boston_tea_party, Label = event


/tmp/ipykernel_3335773/127152020.py:11: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_node, node)


[INFO] Adding Node 2/31011: ID = griffins_wharf, Label = location
[INFO] Adding Node 3/31011: ID = hutchinson_street, Label = location
[INFO] Adding Node 4/31011: ID = val_342, Label = value_number
[INFO] Adding Node 5/31011: ID = pearl_street, Label = location
[INFO] Adding Node 6/31011: ID = samuel_adams, Label = person
[INFO] Adding Node 7/31011: ID = boston_tea_party, Label = event
[INFO] Adding Node 8/31011: ID = val_principled_protest, Label = value_text


[INFO] Adding Node 9/31011: ID = val_constitutional_rights, Label = value_text
[INFO] Adding Node 10/31011: ID = constitution, Label = other
[INFO] Adding Node 11/31011: ID = great_britain, Label = location
[INFO] Adding Node 12/31011: ID = bill_of_rights_1689, Label = law
[INFO] Adding Node 13/31011: ID = parliament, Label = organization
[INFO] Adding Node 14/31011: ID = thomas_hutchinson, Label = person
[INFO] Adding Node 15/31011: ID = london, Label = location
[INFO] Adding Node 16/31011: ID = sons_of_liberty, Label = organization
[INFO] Adding Node 17/31011: ID = dartmouth, Label = other
[INFO] Adding Node 18/31011: ID = eleanor, Label = other
[INFO] Adding Node 19/31011: ID = beaver, Label = other
[INFO] Adding Node 20/31011: ID = ship_owners_and_captains, Label = other
[INFO] Adding Node 21/31011: ID = colonists, Label = organization
[INFO] Adding Node 22/31011: ID = britain, Label = location
[INFO] Adding Node 23/31011: ID = politicians_friends_of_colonies, Label = person
[INFO]

/tmp/ipykernel_3335773/127152020.py:16: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_relationship, relationship)


[INFO] Adding Relationship 2/25702: Type = INVOLVED_COUNT, Source = boston_tea_party, Target = val_342
[INFO] Adding Relationship 3/25702: Type = LOCATED_NEAR, Source = griffins_wharf, Target = hutchinson_street
[INFO] Adding Relationship 4/25702: Type = CURRENTLY_KNOWN_AS, Source = hutchinson_street, Target = pearl_street
[INFO] Adding Relationship 5/25702: Type = DEFENDED, Source = samuel_adams, Target = boston_tea_party
[INFO] Adding Relationship 6/25702: Type = HAS_NATURE, Source = boston_tea_party, Target = val_principled_protest
[INFO] Adding Relationship 7/25702: Type = DEFENSE_FOR, Source = boston_tea_party, Target = val_constitutional_rights
[INFO] Adding Relationship 8/25702: Type = APPLIES_TO, Source = constitution, Target = great_britain
[INFO] Adding Relationship 9/25702: Type = ESTABLISHED_BY, Source = bill_of_rights_1689, Target = parliament
[INFO] Adding Relationship 10/25702: Type = MUST_REPRESENT, Source = parliament, Target = great_britain
[INFO] Adding Relationship 

In [8]:
 Load the knowledge graph data from a JSON file
json_file_path = "/home/sbhavsar/PoisonedRAG/hybrid_approach/jsons/17_05_2025_knowledge_graph_new_sys.json"  # Path to your JSON file
with open(json_file_path, "r") as file:
    knowledge_graph = json.load(file)

# Store the knowledge graph in Neo4j
try:
    store_knowledge_graph(driver, knowledge_graph)
    print("Knowledge graph stored in Neo4j successfully!")
finally:
    driver.close()

SyntaxError: invalid syntax (3306092957.py, line 1)

In [ ]:
def get_all_node_labels():
    with driver.session() as session:
        result = session.run("CALL db.labels()")
        return [record["label"] for record in result]

In [ ]:
# Example usage
labels = get_all_node_labels()
print("Node Labels:", labels)

/tmp/ipykernel_63519/522634782.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


Node Labels: ['other', 'organization', 'document', 'person', 'financial_term', 'work', 'date', 'keyword', 'law', 'location', 'event', 'keyword_label', 'scientific_term', 'product', 'language', 'title', 'concept', 'character', 'award', 'project', 'facility', 'treatment', 'music', 'economic_term', 'economic_policy', 'policy', 'financial_instrument', 'activity']
